In [2]:
# --- GPU-Accelerated Bitcoin Key Recovery ---
!pip install base58 ecdsa python-Levenshtein tqdm pycryptodome numba

from google.colab import drive
drive.mount('/content/drive')

import hashlib, base58, binascii
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from Levenshtein import distance as lev_distance
import numpy as np
from tqdm.auto import tqdm
from numba import cuda
import time

TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
BASE_TIMESTAMP = 1298975179.002815950
FULL_TXID = "e67a0550848b7932d7796aeea16ab0e48a5cfe81c4e8cca2c5b03e0416850114"
RANGE = 0.0001
STEP_SIZE = 1e-9

log_file = '/content/drive/MyDrive/gpu_best_matches.txt'

# GPU-accelerated SHA-256 hashing
@cuda.jit
def gpu_sha256(batch_seeds, batch_hashes):
    idx = cuda.grid(1)
    if idx < batch_seeds.shape[0]:
        data = batch_seeds[idx]
        hash_result = hashlib.sha256(data).digest()
        for i in range(32):
            batch_hashes[idx, i] = hash_result[i]

# Private key to Bitcoin address
def private_key_to_address(privkey_hex):
    privkey_bytes = binascii.unhexlify(privkey_hex)
    sk = SigningKey.from_string(privkey_bytes, curve=SECP256k1)
    vk = sk.get_verifying_key()
    pubkey = b'\x04' + vk.to_string()
    sha = hashlib.sha256(pubkey).digest()
    ripe = RIPEMD160.new(sha).digest()
    payload = b'\x00' + ripe
    checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    address = base58.b58encode(payload + checksum).decode()
    return address

# Prepare batches for GPU
timestamps = np.arange(BASE_TIMESTAMP - RANGE, BASE_TIMESTAMP + RANGE, STEP_SIZE)
total = len(timestamps)
batch_size = 4096  # Optimized for A100 GPU
best_distance = 100
best_result = ""

start_time = time.time()
print("🚀 Starting GPU-Accelerated Search (A100)...")

for start in tqdm(range(0, total, batch_size), desc="Batch Processing"):
    end = min(start + batch_size, total)
    seeds_batch = np.array([f"mtgox-tx-{ts:.12f}-{FULL_TXID}".encode('utf-8') for ts in timestamps[start:end]])

    # Allocate GPU memory
    d_seeds = cuda.to_device(seeds_batch)
    d_hashes = cuda.device_array((len(seeds_batch), 32), dtype=np.uint8)

    # Launch GPU kernel
    threads_per_block = 256
    blocks = (len(seeds_batch) + (threads_per_block - 1)) // threads_per_block
    gpu_sha256[blocks, threads_per_block](d_seeds, d_hashes)
    hashes = d_hashes.copy_to_host()

    for i, hash_bytes in enumerate(hashes):
        privkey_hex = hash_bytes.tobytes().hex()
        addr = private_key_to_address(privkey_hex)
        dist = lev_distance(addr, TARGET_ADDRESS)

        if dist < best_distance:
            best_distance = dist
            best_result = f"Distance: {dist}, Address: {addr}, Seed: {seeds_batch[i].decode()}, PrivKey: {privkey_hex}"
            print(f"[{time.ctime()}] {best_result}")

            with open(log_file, 'a') as f:
                f.write(best_result + "\n")

            if dist == 0:
                print("🎉 Exact match found!", best_result)
                exit()

elapsed = time.time() - start_time
print(f"⏱️ Completed in {elapsed/60:.2f} minutes. Best result: {best_result}")


Mounted at /content/drive
🚀 Starting GPU-Accelerated Search (A100)...


Batch Processing:   0%|          | 0/49 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:579: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


TypingError: Failed in cuda mode pipeline (step: nopython frontend)
Unknown attribute 'sha256' of type Module(<module 'hashlib' from '/usr/lib/python3.11/hashlib.py'>)

File "<ipython-input-2-5f7b91a6bfaa>", line 30:
def gpu_sha256(batch_seeds, batch_hashes):
    <source elided>
        data = batch_seeds[idx]
        hash_result = hashlib.sha256(data).digest()
        ^

During: typing of get attribute at <ipython-input-2-5f7b91a6bfaa> (30)

File "<ipython-input-2-5f7b91a6bfaa>", line 30:
def gpu_sha256(batch_seeds, batch_hashes):
    <source elided>
        data = batch_seeds[idx]
        hash_result = hashlib.sha256(data).digest()
        ^


In [3]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# Install Dependencies
!pip install base58 ecdsa python-Levenshtein tqdm pycryptodome

import hashlib, base58, binascii
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
from Levenshtein import distance as lev_distance
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import time

# Configuration
TARGET_ADDRESS = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
BASE_TIMESTAMP = 1298975179.002815950
FULL_TXID = "e67a0550848b7932d7796aeea16ab0e48a5cfe81c4e8cca2c5b03e0416850114"
INITIAL_RANGE = 0.0001
INITIAL_STEP_SIZE = 1e-9
BATCH_SIZE = 8192  # optimized for A100

log_file = '/content/drive/MyDrive/gpu_best_matches.txt'

def private_key_to_address(privkey_hex):
    privkey_bytes = binascii.unhexlify(privkey_hex)
    sk = SigningKey.from_string(privkey_bytes, curve=SECP256k1)
    vk = sk.get_verifying_key()
    pubkey = b'\x04' + vk.to_string()
    sha = hashlib.sha256(pubkey).digest()
    ripe = RIPEMD160.new(sha).digest()
    payload = b'\x00' + ripe
    checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    address = base58.b58encode(payload + checksum).decode()
    return address

def hash_and_check(seed):
    privkey_hex = hashlib.sha256(seed.encode()).hexdigest()
    addr = private_key_to_address(privkey_hex)
    dist = lev_distance(addr, TARGET_ADDRESS)
    return dist, addr, seed, privkey_hex

best_distance = 100
best_result = ""
current_range = INITIAL_RANGE
current_step_size = INITIAL_STEP_SIZE

start_time = time.time()
print("🚀 Starting Adaptive Search (Optimized for A100 Colab)...")

while current_step_size >= 1e-12:
    timestamps = np.arange(BASE_TIMESTAMP - current_range,
                           BASE_TIMESTAMP + current_range,
                           current_step_size)
    total = len(timestamps)

    print(f"\n🔍 Range: ±{current_range}, Step: {current_step_size}, Iterations: {total}")

    with ThreadPoolExecutor(max_workers=32) as executor:
        for i in tqdm(range(0, total, BATCH_SIZE), desc="Batch processing"):
            batch_ts = timestamps[i:i+BATCH_SIZE]
            seeds_batch = [f"mtgox-tx-{ts:.12f}-{FULL_TXID}" for ts in batch_ts]

            results = executor.map(hash_and_check, seeds_batch)

            for dist, addr, seed, privkey_hex in results:
                if dist < best_distance:
                    best_distance = dist
                    best_result = f"[{time.ctime()}] Distance: {dist}, Addr: {addr}, Seed: {seed}, PrivKey: {privkey_hex}"
                    print(best_result)

                    with open(log_file, 'a') as f:
                        f.write(best_result + "\n")

                    if dist == 0:
                        print("🎉 Exact match found!", best_result)
                        exit()

    # Adaptive refinement
    current_range /= 10
    current_step_size /= 10

elapsed = time.time() - start_time
print(f"⏱️ Completed in {elapsed/60:.2f} minutes.")
print("Best result:", best_result)

# Bit-Flipping Fallback if no exact match
print("\n🔄 Bit-flip fallback starting...")
key_int = int(best_result.split('PrivKey: ')[1], 16)
for bit in tqdm(range(256), desc="Bit-flip fallback"):
    flipped_key_int = key_int ^ (1 << bit)
    flipped_key_hex = format(flipped_key_int, '064x')
    addr = private_key_to_address(flipped_key_hex)
    dist = lev_distance(addr, TARGET_ADDRESS)
    if dist < best_distance:
        best_distance = dist
        best_result = f"[BitFlip-{time.ctime()}] Distance: {dist}, Addr: {addr}, Priv: {flipped_key_hex}"
        print(best_result)
        with open(log_file, 'a') as f:
            f.write(best_result + "\n")
        if dist == 0:
            print("🎉 Exact match found by bit-flip!", best_result)
            exit()

print("Search exhausted. Final best:", best_result)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Adaptive Search (Optimized for A100 Colab)...

🔍 Range: ±0.0001, Step: 1e-09, Iterations: 199795


Batch processing:   0%|          | 0/25 [00:00<?, ?it/s]

[Wed Mar 12 09:39:04 2025] Distance: 31, Addr: 1EmxZEJdbaSz95Jb9jjZfQzvVGrwtXVDcZ, Seed: mtgox-tx-1298975179.002716064453-e67a0550848b7932d7796aeea16ab0e48a5cfe81c4e8cca2c5b03e0416850114, PrivKey: eec74da5a480fdbc2d6062da77889fae02a258824bfea3bf33c7a8dd87e65f05

🔍 Range: ±1e-05, Step: 1e-10, Iterations: 200272


Batch processing:   0%|          | 0/25 [00:00<?, ?it/s]


🔍 Range: ±1.0000000000000002e-06, Step: 1.0000000000000001e-11, Iterations: 190735


...


🔍 Range: ±1.0000000000000002e-07, Step: 1.0000000000000002e-12, Iterations: 0


Batch processing: 0it [00:00, ?it/s]

⏱️ Completed in 8.56 minutes.
Best result: [Wed Mar 12 09:39:04 2025] Distance: 31, Addr: 1EmxZEJdbaSz95Jb9jjZfQzvVGrwtXVDcZ, Seed: mtgox-tx-1298975179.002716064453-e67a0550848b7932d7796aeea16ab0e48a5cfe81c4e8cca2c5b03e0416850114, PrivKey: eec74da5a480fdbc2d6062da77889fae02a258824bfea3bf33c7a8dd87e65f05

🔄 Bit-flip fallback starting...


Bit-flip fallback:   0%|          | 0/256 [00:00<?, ?it/s]

[BitFlip-Wed Mar 12 09:47:34 2025] Distance: 29, Addr: 14LjXKS272oHosyNdWkth6uLCDZW9ZG6YM, Priv: eec74da5a480fdbc2d6062da77889fae02a258824bfea3bf33c7a8dd87665f05
[BitFlip-Wed Mar 12 09:47:34 2025] Distance: 27, Addr: 164eVXb2ryxhpHj2Q6jJzQdFjtw5z8E2di, Priv: eec74da5a480fdbc2d6062da77889fae02a258824bfea3bf33c7e8dd87e65f05
Search exhausted. Final best: [BitFlip-Wed Mar 12 09:47:34 2025] Distance: 27, Addr: 164eVXb2ryxhpHj2Q6jJzQdFjtw5z8E2di, Priv: eec74da5a480fdbc2d6062da77889fae02a258824bfea3bf33c7e8dd87e65f05
